In [1]:
!pip install ollama psutil nltk

  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.6 MB ? eta -:--:--
   ---------------------------------------- 1.6/1.6 MB 4.5 MB/s  0:00:00
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
Using cached h11-0.16.0-py3-none-any.whl (37 kB)
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ------------------------------ --------- 1.6/2.1 MB 8.0 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 7.2 MB/s  0:00:00
Using cached annotated

In [3]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [ ]:
import time
import os
import psutil
import pandas as pd
import ollama
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Ensure necessary NLP resources are downloaded
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

# 1. LOAD YOUR KAGGLE DATASET
csv_path = 'datasets/scisumm.csv' 

if not os.path.exists(csv_path):
    raise FileNotFoundError(f"Please make sure the dataset file is named '{csv_path}' and placed in the same directory as this notebook.")

df = pd.read_csv(csv_path)

# Preview the columns to ensure we map text correctly
print("Dataset columns:", df.columns.tolist())

# ScisummNet typically uses columns like 'document' or 'text' for the main paper body.
# We will identify the text column dynamically.
text_column = 'text' if 'text' in df.columns else df.columns[0]

# Sample 30 papers to keep the 1-week timeline realistic and fast
sample_df = df.head(30)

# 2. PROMPT COMPRESSION FUNCTION
def compress_prompt(text):
    if not isinstance(text, str):
        return ""
    stop_words = set(stopwords.words('english'))
    word_tokens = word_tokenize(text)
    compressed_tokens = [w for w in word_tokens if not w.lower() in stop_words]
    return " ".join(compressed_tokens)

# 3. EXPERIMENTAL TESTING ENGINE
results = []

def run_test(pipeline_name, model_name, input_text, paper_id):
    # Ensure text isn't excessively long for an edge model test loop (truncate to first 800 words)
    truncated_text = " ".join(str(input_text).split()[:800])
    
    process = psutil.Process()
    start_mem = process.memory_info().rss / (1024 * 1024) # MB
    start_time = time.time()
    
    try:
        # Requesting a summarization task from the local Ollama instance
        response = ollama.chat(model=model_name, messages=[
            {'role': 'user', 'content': f"Summarize this scientific text in two sentences: {truncated_text}"}
        ])
        output_text = response['message']['content']
    except Exception as e:
        output_text = f"Error during inference: {str(e)}"
        
    end_time = time.time()
    end_mem = process.memory_info().rss / (1024 * 1024) # MB
    
    latency = end_time - start_time
    memory_used = max(0, end_mem - start_mem)
    
    return {
        'Paper_ID': paper_id,
        'Pipeline': pipeline_name,
        'Latency_Sec': round(latency, 3),
        'RAM_Used_MB': round(memory_used, 2),
        'Output_Word_Count': len(output_text.split())
    }

# 4. RUN THE COMPARATIVE EXPERIMENT LOOP
print("\nStarting optimization benchmarks across 30 scientific papers...")

for idx, row in sample_df.iterrows():
    raw_text = row[text_column]
    print(f"Processing Paper {idx + 1}/30...", end="\r")
    
    # Baseline Group: Full Text on Standard Model
    res_base = run_test("Baseline", "phi3", raw_text, idx)
    
    # Pipeline A Group: Full Text on Quantized Model
    res_pipe_a = run_test("Quantized_Model", "phi3:3.8b-mini-4k-instruct-q4_K_M", raw_text, idx)
    
    # Pipeline B Group: Stripped/Compressed Text on Standard Model
    compressed_text = compress_prompt(raw_text)
    res_pipe_b = run_test("Prompt_Compression", "phi3", compressed_text, idx)
    
    results.extend([res_base, res_pipe_a, res_pipe_b])

print("\nAll experiments complete!")

# 5. CONVERT THE LOGGED METRICS INTO A DATAFRAME AND SAVE
results_df = pd.DataFrame(results)
results_df.to_csv("llm_optimization_results.csv", index=False)

# Show a quick snapshot of the generated data inside your notebook
results_df.head(9)

Dataset columns: ['text', 'summary']

Starting optimization benchmarks across 30 scientific papers...
